# Get WhisperTimeSync

In [15]:
!rm -rf WhisperTimeSync
!git clone https://github.com/EtienneAb3d/WhisperTimeSync.git

Cloning into 'WhisperTimeSync'...
remote: Enumerating objects: 248, done.
remote: Counting objects: 100% (85/85), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 248 (delta 55), reused 68 (delta 41), pack-reused 163 (from 1)
Receiving objects: 100% (248/248), 2.75 MiB | 322.00 KiB/s, done.
Resolving deltas: 100% (113/113), done.


# Transcribe

In [16]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor
import librosa
from transformers import pipeline
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
!nvidia-smi


Thu Dec  5 11:47:22 2024       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.107.02             Driver Version: 550.107.02     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A5000               Off |   00000000:01:00.0 Off |                  Off |
| 30%   25C    P8             17W /  230W |   13271MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [17]:
# Load the fine-tuned model and processor
# model = WhisperForConditionalGeneration.from_pretrained("ivrit-ai/whisper-v2-pd1-e1").to(device)
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-large-v2").to(device)
processor = WhisperProcessor.from_pretrained("openai/whisper-large-v2")

model.generation_config.language = "he"

# Define maximum audio segment length
MAX_SEGMENT_LENGTH = 30 * 1000  # 30 seconds in milliseconds



<div dir="rtl">

# ניסויים שמראים את התוצאות של האפשרויות השונות:

1. קטע קצר (פחות מ30 שניות) ללא התאמה לזמנים
2. קטע קצר (פחות מ30 שניות) עם התאמה לזמנים בחלוקה לקטעים באורך כמה מילים - התוצאה היא שהמודל מאבד חלקים שלמים, גם הזמנים שהוא כותב בקפיצות של שניות שלמות (או יותר)  
3. קטע קצר (פחות מ30 שניות) עם התאמה לזמנים בחלוקה למילים
4. קטע ארוך (יותר מ30 שניות) ללא התאמה לזמנים - לא אפשרי
5. קטע ארוך (יותר מ30 שניות) עם התאמה לזמנים בחלוקה לקטעים - נראה שהזמנים לא נכונים, (או שיש דרך אחרת להבין אותם). גם כאן מאבדים הרבה חלקים. בנוסף בסוף הקטע נכנס "תודה רבה"
6. קטע ארוך (יותר מ30 שניות) עם התאמה לזמנים בחלוקה למילים - הזמנים לא נכונים. בנוסף גם כאן נכנס בסוף "תודה רבה"

</div>

In [18]:
#ניסוי שבודק עם קובץ ארוך וחלוקה לזמנים עם וויספר עצמו.
dir = ""
audio_segment, _ = librosa.load(dir + "audio.mp3", sr=16000)
asr = pipeline("automatic-speech-recognition", model=model, tokenizer=processor.tokenizer,
                   feature_extractor=processor.feature_extractor, device_map="auto")


Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


In [19]:
# play the audio_segment[int(:16000*29.9)
import IPython.display as ipd
ipd.Audio(audio_segment[16000*30:16000*29*2], rate=16000)


In [20]:
# short audio and without timestamps
asr(audio_segment[:16000*30-25], return_timestamps=False,
                 generate_kwargs={"language": "<|he|>",
                                  "task": "transcribe"})

{'text': ' וזאת הברכה אשר ברך משה איש האלוהים את בני ישראל לפני מותו'}

In [21]:
# Short audio and with timestamps for segments (few words in each segment)
asr(audio_segment[:16000*30-25], return_timestamps=True,
                 generate_kwargs={"language": "<|he|>",
                                  "task": "transcribe"})

{'text': ' וזאת הברכה אשר ברך משה איש האלוהים את בני ישראל לפני מותו.',
 'chunks': [{'timestamp': (0.0, 15.68),
   'text': ' וזאת הברכה אשר ברך משה איש האלוהים את בני ישראל לפני מותו.'}]}

In [22]:
# Short audio and with timestamps for words
asr(audio_segment[:16000*30-25], return_timestamps="word",
                 generate_kwargs={"language": "<|he|>",
                                  "task": "transcribe"})

{'text': ' וזאת הברכה אשר ברך משה איש האלוהים את בני ישראל לפני מותו. ויומר אדומי, מסיני בא וזרח מסעיר למו.',
 'chunks': [{'text': ' וזאת', 'timestamp': (0.0, 0.92)},
  {'text': ' הברכה', 'timestamp': (0.92, 3.0)},
  {'text': ' אשר', 'timestamp': (3.0, 7.28)},
  {'text': ' ברך', 'timestamp': (7.28, 7.76)},
  {'text': ' משה', 'timestamp': (7.76, 9.08)},
  {'text': ' איש', 'timestamp': (9.08, 9.92)},
  {'text': ' האלוהים', 'timestamp': (9.92, 10.86)},
  {'text': ' את', 'timestamp': (10.86, 11.5)},
  {'text': ' בני', 'timestamp': (11.5, 11.76)},
  {'text': ' ישראל', 'timestamp': (11.76, 12.46)},
  {'text': ' לפני', 'timestamp': (12.46, 14.08)},
  {'text': ' מותו.', 'timestamp': (14.08, 16.08)},
  {'text': ' ויומר', 'timestamp': (16.08, 16.74)},
  {'text': ' אדומי,', 'timestamp': (16.74, 22.12)},
  {'text': ' מסיני', 'timestamp': (22.12, 23.0)},
  {'text': ' בא', 'timestamp': (23.0, 23.52)},
  {'text': ' וזרח', 'timestamp': (23.52, 25.32)},
  {'text': ' מסעיר', 'timestamp': (25.32, 26.1)},

In [23]:
# Try with the same audio but slice the first 0.1 seconds
asr(audio_segment[1600:16000*30-25], return_timestamps=True,
                 generate_kwargs={"language": "<|he|>",
                                  "task": "transcribe"})

{'text': ' וזאת הברכה אשר ברך משה איש האלוהים את בני ישראל לפני מותו. ויומר אדומי מסיני בא וזרח מסעיר למו',
 'chunks': [{'timestamp': (0.0, 15.6),
   'text': ' וזאת הברכה אשר ברך משה איש האלוהים את בני ישראל לפני מותו.'},
  {'timestamp': (15.6, 26.8), 'text': ' ויומר אדומי מסיני בא וזרח מסעיר למו'}]}

In [24]:
# Try with the another slice of the audio (the next 30 seconds)
asr(audio_segment[16000*30:16000*29*2], return_timestamps=True,
                 generate_kwargs={"language": "<|he|>",
                                  "task": "transcribe"})

{'text': ' ואתה מריבות קודש ממינו אש דת למו אף חובב עמים כל קדושיו בידיך והם תוקו לרגליך ייסע מדבר אותך תורה ציבה לנו משה מורשה',
 'chunks': [{'timestamp': (0.0, 2.8), 'text': ' ואתה מריבות קודש'},
  {'timestamp': (3.12, 7.82), 'text': ' ממינו'},
  {'timestamp': (8.0, 10.72), 'text': ' אש דת למו'},
  {'timestamp': (11.1, 14.62), 'text': ' אף חובב עמים'},
  {'timestamp': (14.88, 16.64), 'text': ' כל קדושיו בידיך'},
  {'timestamp': (17.1, 19.34), 'text': ' והם תוקו לרגליך'},
  {'timestamp': (19.58, 20.88), 'text': ' ייסע'},
  {'timestamp': (21.04, 23.1), 'text': ' מדבר אותך'},
  {'timestamp': (23.56, 25.9), 'text': ' תורה ציבה לנו משה'},
  {'timestamp': (26.26, 27.9), 'text': ' מורשה'}]}

In [25]:
# Try long audio without timestamps
try:
    asr(audio_segment, return_timestamps=False,
                    generate_kwargs={"language": "<|he|>",
                                    "task": "transcribe"})
except Exception as e:
    print(e)

You have passed more than 3000 mel input features (> 30 seconds) which automatically enables long-form generation which requires the model to predict timestamp tokens. Please either pass `return_timestamps=True` or make sure to pass no more than 3000 mel input features.


In [26]:
# Try long audio with timestamps for segments
asr(audio_segment, return_timestamps=True,
                 generate_kwargs={"language": "<|he|>",
                                  "task": "transcribe"})

{'text': " וזאת הברכה אשר ברך משה איש האלוהים את בני ישראל לפני מותו. ויומר אדומי מסיני בה וזרח מסעיר למה הופיע מהר פרן ואתה מריב ובוד קודש ממינו אשדת למה אף חובב עמים כל קדושיו בידיך, והם תוקו לרגליך, ייסע, מדבר אותך. תורה ציווה לנו משה, מורשה, קהילת יעקב, ואיהי וישורון מלך, בהתאסף רשעם, יחד שבטי ישראל, יחיר אובן ועל ימות והיא מתב מספר וזאת ליהודה ויאמר שמע אדוני כל יהודה ואל עמו טבעינו ידיו רב לו ועזר מצריו תהיה ולבי אמר תומך ועורך לאיש חסידך אשר ניסיתו במסע תריווהו על ממריבה. אמר לאביו ולאמו לא ראיתיו, ואת איךיו לא הכיר, ואת בניו לא ידע. כי שמרו אמרתך ובריתך ינצורו, יורו משפטיך ליעקב, ותורתך לישראל ישימו כתורה באפיך וחליל על מזבחיך ברך אדוני חלו ופה על ידיו תרצה מחצמות נעים כמוו ומסנעו מן יקומון לבין ימין אמר ידיד אדוני ישכון לבטח עליו חופף עליו כל היום ובין כתפיו שכן וליוסף אמר מבורכת אדוני ארצו ממגד שמיים מתעל ומתהום רובצת תחת וממגד טבועות שמש וממגד גרש ירחים ומראש הררי קדם וממגד גבעות עולם וממגד ארץ ומלואה ורצון שוכן ייסנה תבוא ת'לה ראש יוסף ולקודקוד נזיר אחיו בכור שורור הדר לו ו

In [27]:
# Try long audio with timestamps for words
asr_res = asr(audio_segment, return_timestamps="word",
                 generate_kwargs={"language": "<|he|>",
                                  "task": "transcribe"})
asr_res

{'text': " וזאת הברכה אשר ברך משה איש האלוהים את בני ישראל לפני מותו. ויומר אדומי מסיני בה וזרח מסעיר למה הופיע מהר פרן ואתה מריב ובוד קודש ממינו אשדת למה אף חובב עמים כל קדושיו בידיך, והם תוקו לרגליך, ייסע, מדבר אותך. תורה ציווה לנו משה, מורשה, קהילת יעקב, ואיהי וישורון מלך, בהתאסף רשעם, יחד שבטי ישראל, יחיר אובן ועל ימות והיא מתב מספר וזאת ליהודה ויאמר שמע אדוני כל יהודה ואל עמו טבעינו ידיו רב לו ועזר מצריו תהיה ולבי אמר תומך ועורך לאיש חסידך אשר ניסיתו במסע תריווהו על ממריבה. אמר לאביו ולאמו לא ראיתיו, ואת איךיו לא הכיר, ואת בניו לא ידע. כי שמרו אמרתך ובריתך ינצורו, יורו משפטיך ליעקב, ותורתך לישראל ישימו כתורה באפיך וחליל על מזבחיך ברך אדוני חלו ופה על ידיו תרצה מחצמות נעים כמוו ומסנעו מן יקומון לבין ימין אמר ידיד אדוני ישכון לבטח עליו חופף עליו כל היום ובין כתפיו שכן וליוסף אמר מבורכת אדוני ארצו ממגד שמיים מתעל ומתהום רובצת תחת וממגד טבועות שמש וממגד גרש ירחים ומראש הררי קדם וממגד גבעות עולם וממגד ארץ ומלואה ורצון שוכן ייסנה תבוא ת'לה ראש יוסף ולקודקוד נזיר אחיו בכור שורור הדר לו ו

In [30]:
# test asr-res (with timestamps in word level)
# play the audio of each word, and print the text of the word
for i, segment in enumerate(asr_res["chunks"]):
    current_chunk_text = segment["text"]
    print(current_chunk_text)
    current_chunk_timestamp_start = int(segment["timestamp"][0]*16000)
    current_chunk_timestamp_end = int(segment["timestamp"][1]*16000)
    if current_chunk_timestamp_end <= current_chunk_timestamp_start: # if the length of the segment is 0 or negative
        print("current_chunk_timestamp_end <= current_chunk_timestamp_start")
        continue
    # play the audio segment
    ipd.display(ipd.Audio(audio_segment[current_chunk_timestamp_start:current_chunk_timestamp_end], rate=16000))
    # wait for the user to press enter to continue. and if the user enters "q" then break the loop
    if input("Press Enter to continue, or 'q' to quit") == "q":
        break
    
    

 וזאת
current_chunk_timestamp_end <= current_chunk_timestamp_start
 הברכה
current_chunk_timestamp_end <= current_chunk_timestamp_start
 אשר
current_chunk_timestamp_end <= current_chunk_timestamp_start
 ברך
current_chunk_timestamp_end <= current_chunk_timestamp_start
 משה
current_chunk_timestamp_end <= current_chunk_timestamp_start
 איש
current_chunk_timestamp_end <= current_chunk_timestamp_start
 האלוהים
current_chunk_timestamp_end <= current_chunk_timestamp_start
 את
current_chunk_timestamp_end <= current_chunk_timestamp_start
 בני
current_chunk_timestamp_end <= current_chunk_timestamp_start
 ישראל
current_chunk_timestamp_end <= current_chunk_timestamp_start
 לפני
current_chunk_timestamp_end <= current_chunk_timestamp_start
 מותו.
current_chunk_timestamp_end <= current_chunk_timestamp_start
 ויומר


 אדומי


 מסיני


 בה


 וזרח


 מסעיר


 למה


 הופיע
